# Observation-masked NPE — Case study I: Zika virus in French Polynesia

Reproduces the French Polynesia results of:

> Retkute R. & Gilligan C.A. *Observation-masked neural posterior estimation for
> heterogeneous epidemiological surveillance data.*

| Output | Paper item |
|---|---|
| `figure1.pdf` | Figure 1 — posterior-predictive fits, six archipelagos |
| `figure2.pdf` | Figure 2 — sequential inference of $R_0$ |
| `table3.csv` | Table 3 — $R_0$ compared with Kucharski *et al.* (2016) |
| `figure_si1.pdf` | SI Figure 1 — posteriors vs population size |
| `figure_si2a.pdf` | SI Figure 2A — simulation-based calibration rank CDFs |
| `figure_si2b.pdf` | SI Figure 2B — TARP coverage |

**Model.** Effective SEIR model of Kucharski *et al.* (2016) with a Binomial
reporting process (paper eqns 2.10–2.20). Ten parameters are inferred:
$\theta = (R_0, p_\mathrm{rep}, p_\mathrm{immune}, p_\mathrm{risk}, s_\mathrm{amp},
s_\mathrm{peak}, D_\mathrm{inc}, D_\mathrm{inf}, I_0, t_0)$.

**Observation masking.** Each record is mapped onto a common 25-week grid and
paired with a binary mask, giving the 51-dimensional network input of paper
eqn 3.1: `[y_1..y_25, m_1..m_25, N/N_max]`.

**Data.** `data/S1_Dataset.csv` — weekly sentinel counts and active-site counts
per archipelago, from Kucharski *et al.* (2016).

**Runtime.** Simulation and training take roughly 90 minutes on a single CPU;
the calibration diagnostics add a few minutes. Set `QUICK = True` for a fast
smoke test (results will not match the paper).

## 1. Imports and configuration

In [ ]:
import time
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats as sstats
from scipy.stats import truncnorm as sp_truncnorm
from sbi.inference import SNPE
from sbi.utils import BoxUniform

warnings.filterwarnings("ignore")

QUICK = False   # True -> tiny training/diagnostic budgets for a smoke test

# --- time grid -----------------------------------------------------------------
T_MAX = 25            # observation window (weeks)
T0_MAX = 10           # maximum epidemic-onset delay (weeks)
T_TOTAL = T_MAX + T0_MAX
N_SUB = 10            # Euler substeps per week

# --- population ----------------------------------------------------------------
N_MIN, N_MAX = 1_000, 250_000
N_SCALE = 250_000.0

# --- inference budgets ---------------------------------------------------------
N_SIM = 8_000 if QUICK else 1_000_000    # training simulations
N_POST = 1_000                            # posterior draws per observed record
N_PPC = 500                               # posterior-predictive draws per panel
N_SBC = 200 if QUICK else 10_000          # SBC / TARP trials
L_SBC = 200                               # posterior draws per SBC trial
CHUNK = 500                               # batch size for chunked SBC/TARP sampling
BATCH = 50_000                            # simulation batch size

SEED = 42

# --- parameters ----------------------------------------------------------------
PARAMS = ["R0", "p_rep", "p_immune", "p_risk", "s_amp", "s_peak",
          "d_inc", "d_inf", "initI", "t0"]
N_PARAMS = len(PARAMS)
INPUT_DIM = 2 * T_MAX + 1     # cases (25) | mask (25) | N/N_scale (1)

# BoxUniform support for the neural posterior estimator: a bounding box covering
# the effective support of the prior in `sample_prior` below.
LOW_BOX = np.array([0.0, 0.0, 0.0, 0.10, 0.0, 10.0, 1.5, 0.20, 0.0, 0.0])
HIGH_BOX = np.array([25.0, 1.0, 0.35, 1.0, 1.0, 30.0, 3.6, 1.25, 80.0, 10.0])

PARAM_LABELS = {
    "R0": r"$R_0$", "p_rep": r"$p_{rep}$", "p_immune": r"$p_{immune}$",
    "p_risk": r"$p_{risk}$", "s_amp": r"$s_{amp}$", "s_peak": r"$s_{peak}$ (wk)",
    "d_inc": r"$D_{inc}$ (wk)", "d_inf": r"$D_{inf}$ (wk)",
    "initI": r"$I_0$", "t0": r"$t_0$ (wk)",
}

FIG_STYLE = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "pdf.fonttype": 42, "ps.fonttype": 42,
}

# Sequential single-hue blue for posterior-predictive magnitude; near-black for data.
COL_95, COL_50, COL_MEDIAN, COL_OBS = "#9ec5f4", "#2a78d6", "#104281", "#0b0b0b"
COL_PEAK = "#e34948"

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

print(f"N_PARAMS={N_PARAMS}  INPUT_DIM={INPUT_DIM}  N_SIM={N_SIM:,}  QUICK={QUICK}")

## 2. Prior

Priors follow Kucharski *et al.* (2016); ranges are listed in SI Table S1.

In [ ]:
_a_inc = (0.0 - 17.8 / 7) / (2.3 / 7)   # truncated-normal lower bound, incubation
_a_inf = (0.0 - 4.7 / 7) / (1.2 / 7)    # truncated-normal lower bound, infectious

def sample_prior(n, rng_np):
    """Draw n parameter vectors from the prior.

    Columns: R0, p_rep, p_immune, p_risk, s_amp, s_peak, d_inc, d_inf, initI, t0.
    """
    seed = int(rng_np.integers(2 ** 30))
    d_inc = sp_truncnorm.rvs(_a_inc, np.inf, loc=17.8 / 7, scale=2.3 / 7,
                             size=n, random_state=seed)
    d_inf = sp_truncnorm.rvs(_a_inf, np.inf, loc=4.7 / 7, scale=1.2 / 7,
                             size=n, random_state=seed + 1)
    return np.column_stack([
        rng_np.uniform(0.0, 25.0, n),                        # R0
        rng_np.uniform(0.0, 1.0, n),                         # p_rep
        np.clip(rng_np.exponential(0.06, n), 0.0, 1.0),      # p_immune
        rng_np.uniform(0.10, 1.0, n),                        # p_risk
        rng_np.uniform(0.0, 1.0, n),                         # s_amp
        rng_np.normal(20.0, 2.0, n),                         # s_peak (weeks)
        d_inc,                                               # D_inc (weeks)
        d_inf,                                               # D_inf (weeks)
        rng_np.exponential(10.0, n),                         # I_0
        rng_np.uniform(0.0, 10.0, n),                        # t_0
    ])

print(pd.DataFrame(sample_prior(50_000, rng), columns=PARAMS)
        .describe().loc[["mean", "std", "min", "max"]].round(3))

## 3. Simulator: effective SEIR with Binomial reporting

$$\dot S = -\beta(t)\,SI/(N p_\mathrm{risk}), \quad
   \dot E = \beta(t)\,SI/(N p_\mathrm{risk}) - E/D_\mathrm{inc}, \quad
   \dot I = E/D_\mathrm{inc} - I/D_\mathrm{inf},$$
$$\dot R = I/D_\mathrm{inf}, \quad \dot Z = E/D_\mathrm{inc},$$

with seasonal forcing
$\beta(t) = (R_0/D_\mathrm{inf})\max\!\left(1 + s_\mathrm{amp}\cos(2\pi(t-s_\mathrm{peak})/52),\,0\right)$
and reporting $Y_t \mid Z_t \sim \mathrm{Binomial}(\mathrm{round}(Z_t), p_\mathrm{rep})$.
The 25-week observation window is a slice beginning at the onset delay $t_0$.

In [ ]:
def seir_batch(theta, N_arr, t_max=T_MAX, t_total=T_TOTAL, n_sub=N_SUB):
    """Vectorised SEIR simulator with Binomial reporting.

    Parameters
    ----------
    theta : (bs, 10) array
        Rows are [R0, p_rep, p_immune, p_risk, s_amp, s_peak, d_inc, d_inf, initI, t0].
    N_arr : (bs,) array
        Population size per simulation.

    Returns
    -------
    (bs, t_max) array
        Per-capita weekly reported cases over the observation window.
    """
    bs = theta.shape[0]
    R0, p_rep, p_imm, p_risk, s_amp, s_peak, d_inc, d_inf, initI, t0_arr = \
        [theta[:, i] for i in range(10)]

    N_risk = N_arr * p_risk
    I0 = np.clip(initI, 0.0, np.maximum(N_risk - 1.0, 0.0))
    R_init = N_risk * p_imm
    S = np.maximum(N_risk - I0 - R_init, 0.0)
    E = np.zeros(bs)
    I = I0.copy()

    dt = 1.0 / n_sub
    weekly_Z = np.zeros((bs, t_total))

    for t_week in range(t_total):
        Z_week = np.zeros(bs)
        for s in range(n_sub):
            t_now = t_week + s * dt
            season = np.maximum(
                1.0 + s_amp * np.cos(2.0 * np.pi * (t_now - s_peak) / 52.0), 0.0)
            beta_eff = R0 / d_inf * season

            new_E = beta_eff * I / N_risk * S
            leave_E = E / d_inc
            leave_I = I / d_inf

            S = np.maximum(S - new_E * dt, 0.0)
            E = np.maximum(E + (new_E - leave_E) * dt, 0.0)
            I = np.maximum(I + (leave_E - leave_I) * dt, 0.0)
            Z_week += leave_E * dt
        weekly_Z[:, t_week] = Z_week

    Z_int = np.maximum(np.round(weekly_Z).astype(int), 0)
    obs = rng.binomial(Z_int, p_rep[:, None])

    t0_int = np.clip(np.round(t0_arr).astype(int), 0, t_total - t_max)
    result = np.zeros((bs, t_max))
    for i in range(bs):
        result[i] = obs[i, t0_int[i]:t0_int[i] + t_max]
    return result / np.maximum(N_arr[:, None], 1.0)


def build_input(cases_pc, mask, N, t_max=T_MAX):
    """Observation-masked network input: [cases * mask, mask, N / N_SCALE]."""
    c = np.asarray(cases_pc[:t_max], float) * mask[:t_max]
    m = np.asarray(mask[:t_max], float)
    return np.concatenate([c, m, [N / N_SCALE]])


assert len(build_input(np.zeros(T_MAX), np.ones(T_MAX), 1e5)) == INPUT_DIM
_check = seir_batch(sample_prior(4, rng), np.array([183645., 31871., 16191., 6310.]))
print("simulator output shape:", _check.shape,
      "| max per-capita reported:", _check.max(axis=1).round(5))

## 4. Surveillance data

Weekly counts of Zika-like illness reported by sentinel general practitioners in
six archipelagos, 11 October 2013 – 28 March 2014. A week is unobserved
($m_t = 0$) when no sentinel practice in that archipelago submitted a return
(paper Table 1).

In [ ]:
ARCHS = {
    "Tahiti":       {"N": 183_645, "sites": "tahitiSites",    "cases": "tahiti"},
    "Sous-le-vent": {"N": 31_871,  "sites": "ile_sousSites",  "cases": "ile_sous"},
    "Moorea":       {"N": 16_191,  "sites": "mooreaSites",    "cases": "moorea"},
    "Tuamotu":      {"N": 16_888,  "sites": "tuamotuSites",   "cases": "tuamotu"},
    "Marquises":    {"N": 8_632,   "sites": "marquisesSites", "cases": "marquises"},
    "Australes":    {"N": 6_310,   "sites": "australesSites", "cases": "australes"},
}

df = pd.read_csv("data/S1_Dataset.csv")

obs_data = {}
for arch, info in ARCHS.items():
    cases = df[info["cases"]].values.astype(float)
    mask = (df[info["sites"]].values > 0).astype(float)
    obs_data[arch] = {"cases": cases, "mask": mask, "N": float(info["N"])}
    missing = np.where(mask == 0)[0] + 1
    print(f"{arch:13s} N={info['N']:>7,}  n_obs={int(mask.sum()):>2d}  "
          f"peak_week={int(np.argmax(cases)):>2d}  peak_cases={int(cases.max()):>3d}  "
          f"missing_weeks={list(missing) if len(missing) else 'none'}")

## 5. Training simulations

For each simulation the number of observed weeks $n_\mathrm{obs}$ is drawn
uniformly from $\{1,\dots,25\}$ and the observed positions uniformly at random
without replacement, so the estimator sees the full space of missingness
patterns and is amortised over arbitrary observation schedules.

In [ ]:
n_batch = max(1, N_SIM // BATCH)
batch_size = N_SIM // n_batch

all_theta, all_x = [], []
t_start = time.time()

for b in range(n_batch):
    theta_b = sample_prior(batch_size, rng)
    N_b = rng.integers(N_MIN, N_MAX + 1, size=batch_size).astype(float)
    obs_b = seir_batch(theta_b, N_b)

    n_obs_b = rng.integers(1, T_MAX + 1, batch_size)
    masks_b = np.zeros((batch_size, T_MAX))
    for i in range(batch_size):
        masks_b[i, rng.choice(T_MAX, n_obs_b[i], replace=False)] = 1.0

    all_theta.append(theta_b)
    all_x.append(np.vstack([build_input(obs_b[i], masks_b[i], N_b[i])
                            for i in range(batch_size)]))
    if (b + 1) % 5 == 0 or b + 1 == n_batch:
        print(f"  batch {b + 1}/{n_batch}  {time.time() - t_start:.1f}s")

theta_all = np.vstack(all_theta)
x_all = np.vstack(all_x)
print(f"\n{len(theta_all):,} simulations in {time.time() - t_start:.1f}s  "
      f"theta {theta_all.shape}  x {x_all.shape}")

## 6. Train the amortised posterior estimator

A Masked Autoregressive Flow trained with SNPE-C, in a single round.

In [ ]:
prior = BoxUniform(low=torch.tensor(LOW_BOX, dtype=torch.float32),
                   high=torch.tensor(HIGH_BOX, dtype=torch.float32))

inference = SNPE(prior=prior)
inference.append_simulations(torch.tensor(theta_all, dtype=torch.float32),
                             torch.tensor(x_all, dtype=torch.float32))

t_train = time.time()
density_estimator = inference.train()
print(f"Training completed in {(time.time() - t_train) / 60:.1f} min")

posterior = inference.build_posterior(density_estimator)
torch.save(density_estimator, "zika_density_estimator.pt")

The saved estimator makes the sections below reproducible without repeating
training. To resume from it, run sections 1–4 and then:

```python
from sbi.inference.posteriors import DirectPosterior
density_estimator = torch.load("zika_density_estimator.pt", weights_only=False)
posterior = DirectPosterior(posterior_estimator=density_estimator, prior=prior)
```

## 7. Posteriors for the six archipelagos

One trained estimator, evaluated on each archipelago's own observation mask,
case series and population size.

In [ ]:
t_infer = time.time()
posterior_samples = {}
rows = []

for arch, d in obs_data.items():
    x_obs = build_input(d["cases"] / d["N"], d["mask"], d["N"])
    samp = posterior.sample(
        (N_POST,), x=torch.tensor(x_obs, dtype=torch.float32).unsqueeze(0),
        show_progress_bars=False).numpy()
    posterior_samples[arch] = samp

    row = {"arch": arch, "N": int(d["N"]), "n_obs": int(d["mask"].sum())}
    for j, p in enumerate(PARAMS):
        lo, med, hi = np.percentile(samp[:, j], [5, 50, 95])
        row[f"{p}_lo"], row[f"{p}_med"], row[f"{p}_hi"] = lo, med, hi
    lo95, hi95 = np.percentile(samp[:, PARAMS.index("p_rep")], [2.5, 97.5])
    row["p_rep_lo95"], row["p_rep_hi95"] = lo95, hi95
    rows.append(row)

rdf = pd.DataFrame(rows)
print(f"Posterior evaluation for all six archipelagos: {time.time() - t_infer:.2f}s\n")
print(rdf[["arch", "N", "n_obs", "R0_med", "p_rep_med", "p_immune_med",
           "p_risk_med", "d_inc_med", "d_inf_med", "s_peak_med", "t0_med"]]
      .round(3).to_string(index=False))
rdf.round(4).to_csv("zika_posterior_summary.csv", index=False)

## 8. Table 3 — $R_0$ compared with Kucharski *et al.* (2016)

In [ ]:
KUCHARSKI_R0 = {          # median (95% CrI) reported by Kucharski et al. (2016)
    "Tahiti": (3.5, 2.6, 5.3),
    "Sous-le-vent": (4.1, 3.1, 5.7),
    "Moorea": (4.8, 3.2, 8.4),
    "Tuamotu": (3.0, 2.2, 6.1),
    "Marquises": (2.6, 1.7, 5.3),
    "Australes": (3.1, 2.2, 4.6),
}

table3 = []
for arch in ARCHS:
    s = posterior_samples[arch][:, PARAMS.index("R0")]
    lo, med, hi = np.percentile(s, [2.5, 50, 97.5])
    k_med, k_lo, k_hi = KUCHARSKI_R0[arch]
    table3.append({
        "archipelago": arch,
        "this_study": f"{med:.1f} ({lo:.1f}-{hi:.1f})",
        "kucharski_2016": f"{k_med:.1f} ({k_lo:.1f}-{k_hi:.1f})",
        "median_difference": round(med - k_med, 2),
    })

table3 = pd.DataFrame(table3)
print(table3.to_string(index=False))
table3.to_csv("table3.csv", index=False)

## 9. Figure 1 — posterior-predictive fits

In [ ]:
with mpl.rc_context(FIG_STYLE):
    fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4), constrained_layout=True)
    weeks = np.arange(T_MAX)
    handles = None

    for i, (ax, (arch, d)) in enumerate(zip(axes.flatten(), obs_data.items())):
        row, col = divmod(i, 3)
        x_obs = build_input(d["cases"] / d["N"], d["mask"], d["N"])
        samp = posterior.sample(
            (N_PPC,), x=torch.tensor(x_obs, dtype=torch.float32).unsqueeze(0),
            show_progress_bars=False).numpy()
        pp = seir_batch(samp, np.full(N_PPC, d["N"], dtype=float)) * d["N"]

        h95 = ax.fill_between(weeks, np.percentile(pp, 2.5, axis=0),
                              np.percentile(pp, 97.5, axis=0), color=COL_95,
                              linewidth=0, label="95% posterior-predictive")
        h50 = ax.fill_between(weeks, np.percentile(pp, 25, axis=0),
                              np.percentile(pp, 75, axis=0), color=COL_50,
                              linewidth=0, label="50% posterior-predictive")
        hmed, = ax.plot(weeks, np.median(pp, axis=0), color=COL_MEDIAN, lw=1.0,
                        label="Posterior-predictive median")
        hobs, = ax.plot(weeks, np.where(d["mask"] > 0, d["cases"], np.nan), "o",
                        ms=3, mfc=COL_OBS, mec="white", mew=0.4, lw=0,
                        label="Observed cases")

        ax.set_title(arch, fontsize=7, pad=3)
        ax.text(-0.18, 1.06, "abcdef"[i], transform=ax.transAxes,
                fontsize=9, fontweight="bold", va="top")
        ax.set_xlim(0, T_MAX - 1)
        ax.set_xticks(np.arange(0, T_MAX, 5))
        ax.spines[["top", "right"]].set_visible(False)
        ax.margins(y=0.08)
        if col == 0:
            ax.set_ylabel("Reported cases per week")
        if row == 1:
            ax.set_xlabel("Week")
        if i == 0:
            handles = [h95, h50, hmed, hobs]

    fig.legend(handles, [h.get_label() for h in handles], loc="lower center",
               ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06), fontsize=6.5,
               handlelength=1.4, columnspacing=1.2)
    fig.savefig("figure1.pdf", bbox_inches="tight")
    fig.savefig("figure1.png", dpi=600, bbox_inches="tight")
    plt.show()

## 10. Sequential inference

The reporting history is replayed as if observed in real time: for
$W = 3,\dots,25$ the first $W$ weeks are retained and all later weeks are
masked as not yet occurred. Each call re-uses the same trained estimator, so no
retraining or resimulation is required.

In [ ]:
WEEKS_SEEN = np.arange(3, T_MAX + 1)

t_seq = time.time()
seq_data = {}
for arch, d in obs_data.items():
    cases_pc = d["cases"] / d["N"]
    seq_rows = []
    for W in WEEKS_SEEN:
        mask_w = d["mask"].copy()
        mask_w[W:] = 0.0
        x_w = build_input(cases_pc, mask_w, d["N"])
        samp = posterior.sample(
            (N_POST,), x=torch.tensor(x_w, dtype=torch.float32).unsqueeze(0),
            show_progress_bars=False).numpy()
        r = {"weeks_seen": int(W)}
        for p in ("R0", "p_rep", "t0"):
            s = samp[:, PARAMS.index(p)]
            r[f"{p}_med"] = float(np.median(s))
            r[f"{p}_lo"] = float(np.percentile(s, 5))
            r[f"{p}_hi"] = float(np.percentile(s, 95))
        seq_rows.append(r)
    seq_data[arch] = {"df": pd.DataFrame(seq_rows),
                      "peak_wk": int(np.argmax(d["cases"]))}

print(f"{len(ARCHS) * len(WEEKS_SEEN)} sequential posterior evaluations in "
      f"{time.time() - t_seq:.1f}s")
pd.concat([d["df"].assign(arch=a) for a, d in seq_data.items()]) \
  .to_csv("zika_sequential_posteriors.csv", index=False)

## 11. Figure 2 — sequential inference of $R_0$

In [ ]:
with mpl.rc_context(FIG_STYLE):
    fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4), constrained_layout=True)
    handles = None

    for i, (ax, (arch, d)) in enumerate(zip(axes.flatten(), seq_data.items())):
        seq_df, peak_wk = d["df"], d["peak_wk"]
        final_val = seq_df.loc[seq_df["weeks_seen"] == T_MAX, "R0_med"].values[0]

        h95 = ax.fill_between(seq_df["weeks_seen"], seq_df["R0_lo"], seq_df["R0_hi"],
                              color=COL_95, linewidth=0,
                              label="5-95% posterior interval")
        hmed, = ax.plot(seq_df["weeks_seen"], seq_df["R0_med"], color=COL_MEDIAN,
                        lw=1.2, marker="o", ms=2.5, label="Posterior median")
        hfin = ax.axhline(final_val, color="#0b0b0b", ls="--", lw=1.0,
                          label="Full-series estimate")
        hpk = ax.axvline(peak_wk, color=COL_PEAK, ls=":", lw=1.0, label="Peak")

        row, col = divmod(i, 3)
        ax.set_title(arch, fontsize=7, pad=3)
        ax.set_xlim(WEEKS_SEEN.min(), WEEKS_SEEN.max())
        ax.spines[["top", "right"]].set_visible(False)
        ax.margins(y=0.08)
        if col == 0:
            ax.set_ylabel(r"$R_0$")
        if row == 1:
            ax.set_xlabel("Weeks of surveillance data available")
        if i == 0:
            handles = [h95, hmed, hfin, hpk]

    fig.legend(handles, [h.get_label() for h in handles], loc="lower center",
               ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06), fontsize=6.5,
               handlelength=1.4, columnspacing=1.2)
    fig.savefig("figure2.pdf", bbox_inches="tight")
    fig.savefig("figure2.png", dpi=600, bbox_inches="tight")
    plt.show()

## 12. SI Figure 1 — posteriors vs population size

In [ ]:
SITE_COLORS = ["#2a78d6", "#008300", "#e87ba4", "#eda100", "#1baf7a", "#eb6834"]
SITE_MARKERS = ["o", "s", "^", "D", "v", "P"]

rdf_sorted = rdf.sort_values("N").reset_index(drop=True)
names = rdf_sorted["arch"].tolist()
Ns = np.asarray(rdf_sorted["N"].tolist())

with mpl.rc_context({**FIG_STYLE, "font.size": 7, "axes.labelsize": 8,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5}):
    nrow, ncol = 2, 5
    fig, axes = plt.subplots(nrow, ncol, figsize=(12.0, 5.2), constrained_layout=True)
    axes = axes.ravel()
    legend_handles = None

    for i, (ax, p) in enumerate(zip(axes, PARAMS)):
        med = rdf_sorted[f"{p}_med"].values
        lo = rdf_sorted[f"{p}_lo"].values
        hi = rdf_sorted[f"{p}_hi"].values
        handles = [
            ax.errorbar(Ns[j], med[j],
                        yerr=[[med[j] - lo[j]], [hi[j] - med[j]]],
                        fmt=SITE_MARKERS[j], ms=5.5, capsize=2.5, elinewidth=1.0,
                        color=SITE_COLORS[j], mec="#0b0b0b", mew=0.4,
                        label=name, zorder=3)
            for j, name in enumerate(names)
        ]
        if legend_handles is None:
            legend_handles = handles
        ax.set_xscale("log")
        ax.set_ylabel(PARAM_LABELS[p])
        if i // ncol == nrow - 1:
            ax.set_xlabel("Population size, N")
        ax.spines[["top", "right"]].set_visible(False)
        ax.margins(y=0.15)

    fig.legend(legend_handles, names, loc="lower center", ncol=len(names),
               frameon=False, bbox_to_anchor=(0.5, -0.06), fontsize=7.5,
               handlelength=1.2, columnspacing=1.4)
    fig.savefig("figure_si1.pdf", bbox_inches="tight")
    fig.savefig("figure_si1.png", dpi=600, bbox_inches="tight")
    plt.show()

## 13. Simulation-based calibration

Draw $\theta_k \sim p(\theta)$, simulate $x_k$, and rank each true $\theta_k$
among $L$ posterior draws. Under calibration the ranks are uniform, so the
empirical rank CDF tracks the diagonal within the 95% envelope.

In [ ]:
from sbi.diagnostics import check_sbc, run_sbc

rng_diag = np.random.default_rng(2024)

theta_sbc = sample_prior(N_SBC, rng_diag)
N_sbc = rng_diag.integers(N_MIN, N_MAX + 1, size=N_SBC).astype(float)
obs_sbc = seir_batch(theta_sbc, N_sbc)

n_obs_sbc = rng_diag.integers(1, T_MAX + 1, N_SBC)
masks_sbc = np.zeros((N_SBC, T_MAX))
for i in range(N_SBC):
    masks_sbc[i, rng_diag.choice(T_MAX, n_obs_sbc[i], replace=False)] = 1.0

x_sbc = np.vstack([build_input(obs_sbc[i], masks_sbc[i], N_sbc[i])
                   for i in range(N_SBC)])
theta_t = torch.tensor(theta_sbc, dtype=torch.float32)
x_t = torch.tensor(x_sbc, dtype=torch.float32)

t_sbc = time.time()
rank_chunks, dap_chunks = [], []
for start in range(0, N_SBC, CHUNK):
    end = min(start + CHUNK, N_SBC)
    ranks_c, dap_c = run_sbc(theta_t[start:end], x_t[start:end], posterior,
                             num_posterior_samples=L_SBC, show_progress_bar=False)
    rank_chunks.append(ranks_c)
    dap_chunks.append(dap_c)

ranks = torch.cat(rank_chunks, dim=0)
dap_samples = torch.cat(dap_chunks, dim=0)
print(f"{N_SBC:,} SBC trials in {time.time() - t_sbc:.1f}s")

sbc_stats = check_sbc(ranks, theta_t, dap_samples, num_posterior_samples=L_SBC)
for k, v in sbc_stats.items():
    print(f"  {k:20s}", np.round(v.numpy(), 3))

## 14. SI Figure 2A — rank CDFs

In [ ]:
ranks_np = ranks.numpy()
r_grid = np.arange(L_SBC + 1)
p_null = (r_grid + 1) / (L_SBC + 1)
ci_lo, ci_hi = sstats.binom.interval(0.95, N_SBC, p_null)
ci_lo, ci_hi = ci_lo / N_SBC, ci_hi / N_SBC

nrow, ncol = 2, 5
fig, axes = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.4 * nrow),
                         sharex=True, sharey=True)
axes = axes.ravel()
for ax, p in zip(axes, PARAMS):
    ranks_sorted = np.sort(ranks_np[:, PARAMS.index(p)])
    ecdf = np.searchsorted(ranks_sorted, r_grid, side="right") / N_SBC
    ax.fill_between(r_grid / L_SBC, ci_lo, ci_hi, color="0.5", alpha=0.3, zorder=0)
    ax.plot([0, 1], [0, 1], color="k", lw=1, ls="--", zorder=1)
    ax.plot(r_grid / L_SBC, ecdf, color=COL_50, lw=2, zorder=2)
    ax.set_title(PARAM_LABELS.get(p, p))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Normalised rank")
    ax.set_ylabel("Empirical CDF")
fig.tight_layout()
fig.savefig("figure_si2a.pdf", bbox_inches="tight")
fig.savefig("figure_si2a.png", dpi=300, bbox_inches="tight")
plt.show()

## 15. TARP — joint coverage

SBC checks marginal calibration one parameter at a time. TARP (Lemos *et al.*
2023) checks the full joint posterior, giving expected coverage probability
against credibility level. Positive area-to-curve indicates overdispersion
(conservative intervals), negative indicates underdispersion.

In [ ]:
from sbi.diagnostics.tarp import (_run_tarp, check_tarp,
                                  get_posterior_samples_on_batch,
                                  get_tarp_references)

t_tarp = time.time()
tarp_chunks = []
for start in range(0, N_SBC, CHUNK):
    end = min(start + CHUNK, N_SBC)
    tarp_chunks.append(get_posterior_samples_on_batch(
        x_t[start:end], posterior, (L_SBC,), num_workers=1,
        show_progress_bar=False, use_batched_sampling=True))
tarp_posterior_samples = torch.cat(tarp_chunks, dim=1)

ecp, alpha = _run_tarp(tarp_posterior_samples, theta_t,
                       get_tarp_references(theta_t), num_bins=30,
                       z_score_theta=True)
atc, ks_pval = check_tarp(ecp, alpha)
print(f"TARP in {time.time() - t_tarp:.1f}s")
print(f"  area-to-curve: {atc:+.4f}   (>0 overdispersed, <0 underdispersed)")
print(f"  KS p-value:    {ks_pval:.4f}   (>0.05 consistent with calibration)")

## 16. SI Figure 2B — TARP coverage

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Ideal (perfectly calibrated)")
ax.plot(alpha.numpy(), ecp.numpy(), color=COL_50, lw=2)
ax.set_xlabel(r"Credibility level $\alpha$")
ax.set_ylabel("Expected coverage probability")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig("figure_si2b.pdf", bbox_inches="tight")
fig.savefig("figure_si2b.png", dpi=300, bbox_inches="tight")
plt.show()